# Analysis of 2026 pastry data
The following bakeries were included:
- `Bread Chic`
- `La Creperie`
- `Phillippe`
- `Trickster`
- `Little Bird`

In addition, the following negative controls were used:
- `Lamar donut (regular)`
- `Lamar donut (cake)`
- `King Soopers`
- `Bread (homemade)`

### Metrics
The metrics from the analysis in `2024` are carried over and augmented with a new metric: `spring exponential doughcay`. This is the half-life of an exponential decay curve fit to measured croissant heights following successive squishes with my finger.

The theory is a doughy, underbaked croissant will have a very short half-life (no supporting structure), while a very overbaked, bready croissant will have a very long half-life (lots of spring/structure). An ideal croissant is somewhere in between.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import altair as alt

In [2]:
# load up the data

DATA_PATH = Path().resolve().parent / "data" / "2026" / "croissant_data.csv"
data = pd.read_csv(DATA_PATH)

## Chew

In [3]:
from pytiserie.spring import compute_spring_metrics, exponential, double_exponential

In [4]:
def get_spring_measurements(data: pd.DataFrame) -> pd.DataFrame:
    df = data.loc[data.metric.str.startswith("height_")]
    df['t'] = df.metric.str.split("_").str[-1].str[-1].astype(int)  # starting with t=0
    df = df.sort_values(by='t', ascending=True)
    return df

In [10]:
spring_data = get_spring_measurements(data)

spring_results = {
    "bakery": [],
    "fit_type": [],
    "t": [],
    "y": [],
}

typ = "double"

for (croissant, df) in spring_data.groupby('bakery'):
    double = typ == "double"
    y = df['value'].values
    t = np.arange(len(y))
    
    metrics = compute_spring_metrics(y, double=double)

    # make an interpolant to visualize the fitted curve
    interpolated = np.arange(0, len(df), 0.1)
    func = double_exponential if double else exponential
    fitted_values = func(interpolated, *metrics["params"])
    
    spring_results['bakery'].extend([croissant] * len(interpolated))
    spring_results['fit_type'].extend([typ] * len(interpolated))
    spring_results['t'].extend(interpolated)
    spring_results['y'].extend(fitted_values)

spring_results = pd.DataFrame(spring_results)

In [13]:
base = (
    alt.Chart(spring_data.loc[spring_data['t'].le(8)])
    .transform_window(
        initial_value="first_value(value)",
        sort=[{"field": "t"}],
        groupby=["bakery"],
    )
    .transform_calculate(normalized="datum.value / datum.initial_value")
    .mark_line(size=3, opacity=0.5)
    .encode(
        x='t:Q',
        y='normalized:Q',
        color='bakery'
    )
)

fit = (
    alt.Chart(df)
    .transform_window(
        initial_value="first_value(y)",
        sort=[{"field": "t"}],
        groupby=["bakery"],
    )
    .transform_calculate(normalized="datum.y / datum.initial_value")
    .mark_line(size=2, strokeDash=[5, 5])
    .encode(
        x=alt.X('t:Q', title="# of squishes"),
        y=alt.Y('normalized:Q', title="relative height"),
        color='bakery'
    )
    .properties(title=f"fitted {typ} exponential curve")
)

chart = base + fit
chart.display()

alt.LayerChart(...)

## Color

## Cavity

## Crunch